In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

In [3]:
df = pd.read_csv("../data/raw/T_ONTIME_REPORTING.csv")

In [4]:
df.head()

,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,TAIL_NUM,OP_CARRIER_FL_NUM,ORIGIN_AIRPORT_ID,ORIGIN_AIRPORT_SEQ_ID,ORIGIN_CITY_MARKET_ID,ORIGIN,...,DEST_STATE_ABR,DEST_STATE_NM,CRS_DEP_TIME,DEP_DEL15,DEP_TIME_BLK,CRS_ARR_TIME,CRS_ELAPSED_TIME,FLIGHTS,DISTANCE,DISTANCE_GROUP
0,1,3,5/1/2024 12:00:00 AM,9E,N131EV,4934,13487,1348702,31650,MSP,...,WI,Wisconsin,1950,0.0,1900-1959,2050,60.0,1.0,175.0,1
1,1,3,5/1/2024 12:00:00 AM,9E,N131EV,5047,13487,1348702,31650,MSP,...,PA,Pennsylvania,1245,0.0,1200-1259,1552,127.0,1.0,726.0,3
2,1,3,5/1/2024 12:00:00 AM,9E,N131EV,5047,14122,1412202,30198,PIT,...,MN,Minnesota,1632,0.0,1600-1659,1755,143.0,1.0,726.0,3
3,1,3,5/1/2024 12:00:00 AM,9E,N131EV,5130,11193,1119302,33105,CVG,...,MN,Minnesota,1040,0.0,1000-1059,1142,122.0,1.0,596.0,3
4,1,3,5/1/2024 12:00:00 AM,9E,N131EV,5167,11433,1143302,31295,DTW,...,KY,Kentucky,840,0.0,0800-0859,952,72.0,1.0,229.0,1


In [5]:
print(df["DEP_DEL15"].unique())

[ 0.  1. nan]


In [6]:
# Tipos de datos  por columna
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 609743 entries, 0 to 609742
Data columns (total 27 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   DAY_OF_MONTH           609743 non-null  int64  
 1   DAY_OF_WEEK            609743 non-null  int64  
 2   FL_DATE                609743 non-null  str    
 3   OP_UNIQUE_CARRIER      609743 non-null  str    
 4   TAIL_NUM               608342 non-null  str    
 5   OP_CARRIER_FL_NUM      609743 non-null  int64  
 6   ORIGIN_AIRPORT_ID      609743 non-null  int64  
 7   ORIGIN_AIRPORT_SEQ_ID  609743 non-null  int64  
 8   ORIGIN_CITY_MARKET_ID  609743 non-null  int64  
 9   ORIGIN                 609743 non-null  str    
 10  ORIGIN_CITY_NAME       609743 non-null  str    
 11  ORIGIN_STATE_ABR       609743 non-null  str    
 12  DEST_AIRPORT_ID        609743 non-null  int64  
 13  DEST_AIRPORT_SEQ_ID    609743 non-null  int64  
 14  DEST_CITY_MARKET_ID    609743 non-null  int64  

In [7]:
# Shape y memoria -> HAY 609.743 Vuelos en el mes de mayo del 2024
print("Shape:", df.shape)

Shape: (609743, 27)


In [8]:
# 1. Chequeamos si hay registros nulos en la variable objetivo )dep_del15).
# Eso es un indicador si el vuelo esta cancelado o no
nulos_target = df['DEP_DEL15'].isna().sum()
print(f"Valores nulos en DEP_DEL15: {nulos_target}")

Valores nulos en DEP_DEL15: 7976


In [9]:
# 2. Chequeamos los valores válidos presentes. .value... cuenta la frecuencia de cada valor único en esa columna
print(df['DEP_DEL15'].value_counts(dropna=False))

DEP_DEL15
0.0    447354
1.0    154413
NaN      7976
Name: count, dtype: int64


In [10]:
# Filtramos los nulos de DEP_DEL15 (cancelaciones / no reportados)
df_validos = df.dropna(subset=['DEP_DEL15']).copy()
df_validos['DEP_DEL15'] = df_validos['DEP_DEL15'].astype(int)

print(f"Observaciones antes del filtrado: {len(df):,}")
print(f"Observaciones del universo válido: {len(df_validos):,}")

Observaciones antes del filtrado: 609,743
Observaciones del universo válido: 601,767


In [11]:
#Definición de variables redundantes a eliminar

cols_redundantes = [
    # Varianza cero
    'FLIGHTS',
    # Redundancias en Origen
    'ORIGIN_AIRPORT_ID',
    'ORIGIN_AIRPORT_SEQ_ID',
    'ORIGIN_CITY_NAME',
    # Redundancias en Destino 
    'DEST_AIRPORT_ID',
    'DEST_AIRPORT_SEQ_ID',
    'DEST_CITY_NAME',
    'DEST_STATE_NM',  
]

In [16]:
# Eliminación de variables redundantes
df_processed = df_validos.drop(columns=cols_redundantes)
print(
    f"Se eliminaron {len(cols_redundantes)} columnas redundantes con éxito.\n"
)

Se eliminaron 8 columnas redundantes con éxito.



In [14]:
df_processed.head()

,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,OP_UNIQUE_CARRIER,TAIL_NUM,OP_CARRIER_FL_NUM,ORIGIN_CITY_MARKET_ID,ORIGIN,ORIGIN_STATE_ABR,DEST_CITY_MARKET_ID,DEST,DEST_STATE_ABR,CRS_DEP_TIME,DEP_DEL15,DEP_TIME_BLK,CRS_ARR_TIME,CRS_ELAPSED_TIME,DISTANCE,DISTANCE_GROUP
0,1,3,5/1/2024 12:00:00 AM,9E,N131EV,4934,31650,MSP,MN,30424,CWA,WI,1950,0,1900-1959,2050,60.0,175.0,1
1,1,3,5/1/2024 12:00:00 AM,9E,N131EV,5047,31650,MSP,MN,30198,PIT,PA,1245,0,1200-1259,1552,127.0,726.0,3
2,1,3,5/1/2024 12:00:00 AM,9E,N131EV,5047,30198,PIT,PA,31650,MSP,MN,1632,0,1600-1659,1755,143.0,726.0,3
3,1,3,5/1/2024 12:00:00 AM,9E,N131EV,5130,33105,CVG,KY,31650,MSP,MN,1040,0,1000-1059,1142,122.0,596.0,3
4,1,3,5/1/2024 12:00:00 AM,9E,N131EV,5167,31295,DTW,MI,33105,CVG,KY,840,0,0800-0859,952,72.0,229.0,1


In [17]:
#Casteo de datos
# Asegurar que los IDs de mercado y números de vuelo no se lean como valores numéricos
df_processed['ORIGIN_CITY_MARKET_ID'] = df_processed[
    'ORIGIN_CITY_MARKET_ID'
].astype(str)
df_processed['DEST_CITY_MARKET_ID'] = df_processed[
    'DEST_CITY_MARKET_ID'
].astype(str)

if 'OP_CARRIER_FL_NUM' in df_processed.columns:
  df_processed['OP_CARRIER_FL_NUM'] = df_processed[
      'OP_CARRIER_FL_NUM'
  ].astype(str)

print(f"Dimensiones finales procesadas: {df_processed.shape}")


Dimensiones finales procesadas: (601767, 19)


In [18]:
df_processed.info()


<class 'pandas.DataFrame'>
Index: 601767 entries, 0 to 609742
Data columns (total 19 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   DAY_OF_MONTH           601767 non-null  int64  
 1   DAY_OF_WEEK            601767 non-null  int64  
 2   FL_DATE                601767 non-null  str    
 3   OP_UNIQUE_CARRIER      601767 non-null  str    
 4   TAIL_NUM               601767 non-null  str    
 5   OP_CARRIER_FL_NUM      601767 non-null  str    
 6   ORIGIN_CITY_MARKET_ID  601767 non-null  str    
 7   ORIGIN                 601767 non-null  str    
 8   ORIGIN_STATE_ABR       601767 non-null  str    
 9   DEST_CITY_MARKET_ID    601767 non-null  str    
 10  DEST                   601767 non-null  str    
 11  DEST_STATE_ABR         601767 non-null  str    
 12  CRS_DEP_TIME           601767 non-null  int64  
 13  DEP_DEL15              601767 non-null  int64  
 14  DEP_TIME_BLK           601767 non-null  str    
 15 